In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Bagging Trees for Direct Marketing Response (UCI Bank Marketing)

### Context & Goal

You’re hired by a retail bank to predict whether a customer will subscribe to a term deposit after a marketing call. Your job is to build a **bagging classifier** that beats a single decision tree and to prove the gain is due to **variance reduction**, not luck.

### Learning Targets

* Engineer a full binary-classification pipeline on mixed tabular data (categorical + numeric).
* Train a single Decision Tree vs a Bagging ensemble; quantify bias/variance behaviour.
* Handle **class imbalance** correctly; report metrics that matter (ROC-AUC, PR-AUC, F1).
* Use cross-validation and (optionally) out-of-bag scoring to estimate generalisation.

### Dataset

* **Name:** Bank Marketing
* **Source URL:** [UCI Machine Learning Repository – Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank-marketing)
* **Description:** Marketing calls to Portuguese bank customers with demographics (age, job, education), call details (duration, contact type, month, day), and prior campaign outcomes.
* **Target:** `y` — `yes` if client subscribed, `no` otherwise.
* **Why this fits:** Mixed types, moderate size (~41k rows), **imbalanced** positive class — ideal to test whether bagging stabilises trees and improves decision quality beyond raw accuracy.

### Task Instructions

1. **Data loading & audit**

   * Load the **full** dataset (`bank-full.csv`). Verify row count, dtypes, missing markers (e.g., `"unknown"`), and class ratio.
   * Create a compact data dictionary from the header; flag high-cardinality categoricals if any.

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
bank_marketing = fetch_ucirepo(id=222)

# data (as pandas dataframes)
X = bank_marketing.data.features
y = bank_marketing.data.targets


{'uci_id': 222, 'name': 'Bank Marketing', 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing', 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv', 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 45211, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'], 'target_col': ['y'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2014, 'last_updated': 'Fri Aug 18 2023', 'dataset_doi': '10.24432/C5K306', 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'], 'intro_paper': {'ID': 277, 'type': 'NATIVE', 'title': 'A data-driven approach to predict the s

In [3]:
meta = bank_marketing.metadata
meta_df = pd.DataFrame(
	data=[(k, v) for k, v in meta.items() if v is not None],
	columns=["Attribute", "Value"],
)
display(meta_df)

,Attribute,Value
0,uci_id,222
1,name,Bank Marketing
2,repository_url,https://archive.ics.uci.edu/dataset/222/bank+m...
3,data_url,https://archive.ics.uci.edu/static/public/222/...
4,abstract,The data is related with direct marketing camp...
5,area,Business
6,tasks,[Classification]
7,characteristics,[Multivariate]
8,num_instances,45211
9,num_features,16


In [4]:
X

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,NaN
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,NaN
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,NaN


In [5]:
y

,y
0,no
1,no
2,no
3,no
4,no
...,...
45206,yes
45207,yes
45208,yes
45209,no


2. **Preprocessing choices (justify each)**

   * Encode categoricals (one-hot or target encoding—your call; defend it).
   * Treat `"unknown"` explicitly (keep as level vs impute; explain).
   * Scale numeric features *only if* your downstream choice needs it (trees don’t; pipelines should still be consistent).
   * Split **stratified** into train/test (e.g., 70/30). Fix a seed.


3. **Baseline: single tree**

   * Train a DecisionTreeClassifier. Set depth/leaf constraints you deem reasonable; document them.
   * Evaluate on the **test set** with: ROC-AUC, PR-AUC, accuracy, precision, recall, F1. Include a confusion matrix.
   * Record cross-validated ROC-AUC on the **training split** (e.g., StratifiedKFold, k=5 or 10) to estimate variance (mean ± std).

4. **Bagging ensemble**

   * Train a BaggingClassifier with your decision tree as base estimator. Use bootstrapping and **n_estimators ≥ 50**. Keep other settings explicit (max_samples, max_features).
   * Compute the **same metrics** on the test set.
   * Estimate variability via CV as above; additionally report **OOB score** if you enable `oob_score=True`, and compare OOB vs CV.


5. **Imbalance sensitivity**

   * Re-run (tree and bagging) with a **class-weighting** strategy or a **resampled** training set (e.g., stratified undersample or SMOTE). Report the delta in PR-AUC and recall. Explain which approach you’d ship and why.


6. **Analysis**

   * State clearly whether bagging improved performance. Back it with numbers (ROC-AUC/PR-AUC and the reduction in CV std).
   * Argue **bias vs variance**: did test metrics rise mainly because variance fell (tighter CV distribution) or because you altered bias (systematic error) via reweighting/encoding?


### Evaluation

Success is:

* Reproducible pipeline; decisions justified, not guessed.
* A metric table comparing **Single Tree vs Bagging** (and, if attempted, weighted/resampled variants) with ROC-AUC and PR-AUC as primary.
* Evidence of **variance reduction** (smaller CV std or tighter OOB confidence).
* A short, technical conclusion on deployability (precision–recall trade-off under imbalance).


### Stretch Ideas

* Tune `n_estimators`, `max_samples`, and `bootstrap_features`; plot performance vs ensemble size.
* Compare Bagging with **RandomForest** (feature subsampling) and explain differences in bias/variance.
* Calibrate probabilities (Platt/isotonic) and evaluate **Brier score** + reliability curves.
